# Testing the TSFM evidence and model catalog tools

A hands-on walkthrough of the TSFM tools. Run the cells top to bottom and read each response: this notebook does not assert pass/fail, it shows you what each tool returns so you can compare against what you expect.

| # | tool | what it does |
|---|------|--------------|
| 1 | `list_tasks` | list the standardized TSFM tasks |
| 2 | `profile_series` | summarize a series from a file pointer |
| 3 | `characterize_series` | describe the shape of a series as evidence |
| 4 | `data_quality` | clean a series and report quality stats |

Calls go through **MCPHub / ToolUniverse**, the same path an agent uses:

```text
ToolUniverse --> stdio --> tsfm-mcp-server --> file-pointer I/O
```


## 0. Prerequisites

From the repo root or from this notebook folder, before starting Jupyter:

```bash
uv run python -m ipykernel install --user \
    --name assetopsbench-mcp --display-name "assetopsbench-mcp (uv)"

uv run jupyter lab notebook/model_catalog_tools_model_management_chathurangi.ipynb
```

`--reset` drops the databases first. Only the `default` scenario carries the TSFM catalogs - `scenario_1` / `scenario_2` hold work orders only.

You can watch the data in CouchDB's web UI at http://localhost:5984/_utils (admin/password).


## 1. Prepare a file pointer

We materialize a real telemetry file into a CSV file pointer, then reuse that pointer for the evidence tools.

**Expect:** a `file://...` pointer pointing to a CSV in `/tmp/tsfm_work`.


In [ ]:
import os, sys
from pathlib import Path

# If we are launched from notebook/, REPO should be the parent directory.
cwd = Path.cwd()
if (cwd / "src").exists():
    REPO = cwd
elif (cwd.parent / "src").exists():
    REPO = cwd.parent
else:
    REPO = Path(os.environ.get("AOB_REPO", cwd))

SRC = REPO / "src"
if not SRC.exists():
    raise RuntimeError(f"Could not find repo src/ folder at {SRC}")

sys.path.insert(0, str(SRC))
os.environ["PYTHONPATH"] = str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("repo :", REPO)
print("src  :", SRC)


In [ ]:
dataset_csv, dataset_pointer = materialize_csv_pointer(DEFAULT_DATASET)

# `characterize_series` is expensive on the full telemetry dump, so keep a compact smoke sample.
smoke_csv = dataset_csv.with_name(f"{dataset_csv.stem}_smoke.csv")
df_smoke = pd.read_csv(dataset_csv).head(200).copy()
df_smoke.to_csv(smoke_csv, index=False)
smoke_pointer = smoke_csv.as_uri()

nan_csv, nan_pointer = make_nan_variant(dataset_csv)

print_block("dataset", {"source": str(DEFAULT_DATASET), "csv": str(dataset_csv), "pointer": dataset_pointer})
print_block("smoke_dataset", {"csv": str(smoke_csv), "pointer": smoke_pointer, "rows": len(df_smoke)})
print_block("nan_variant", {"csv": str(nan_csv), "pointer": nan_pointer})


## 1. Connect through MCPHub

`load_tools` spawns the tsfm server as a subprocess and discovers its tools over stdio.


In [ ]:
from mcphub import ToolUniverse

# Run the TSFM server as a subprocess from the current repo checkout.
SERVER_CMD = os.environ.get("SERVER_CMD", f"{sys.executable} -m servers.tsfm.main").split()

tu = ToolUniverse(servers={"tsfm": SERVER_CMD})
n = tu.load_tools(servers=["tsfm"])
print(f"{n} tools discovered\n")
print("\n".join(sorted(tu.all_tools)))


In [ ]:
def ensure_tu():
    global tu
    try:
        # A tiny probe that will fail fast if the session is gone.
        tu.run({"name": "tsfm.list_tasks", "arguments": {}})
    except Exception:
        try:
            tu.close()
        except Exception:
            pass
        tu = ToolUniverse(servers={"tsfm": SERVER_CMD})
        tu.load_tools(servers=["tsfm"] )
        print("Recreated MCP session and reloaded tsfm tools.")
    return tu


## 2. Smoke test the evidence tools

These are the four tools we want to verify first. Each cell is intentionally small and readable, so you can compare the raw tool output against the notebook explanation.

### `list_tasks`

**Input:** `{}`

**Expect:** the catalog of standardized TSFM tasks.

In [ ]:
ensure_tu()
r = tu.run({'name': 'tsfm.list_tasks', 'arguments': {}})
print(json.dumps(r, indent=2, default=str))


### `profile_series`

**Input:** the file pointer above plus `timestamp_column`.

**Expect:** a per-series summary. On a flat telemetry table this should return the main series statistics, and it should not fail when the dataset has multiple numeric columns.

In [ ]:
ensure_tu()
r = tu.run({
    'name': 'tsfm.profile_series',
    "arguments": {
        "dataset_path": dataset_pointer,
        "timestamp_column": "timestamp",
    },
})
print(json.dumps(r, indent=2, default=str))

# Try again with an explicit channel list when the dataset has numeric columns.
df = pd.read_csv(dataset_csv)
channels = [c for c in df.columns if c not in {'timestamp', 'time', 'date'}][:3]
if channels:
    r = tu.run({
        'name': 'tsfm.profile_series',
        "arguments": {
            "dataset_path": dataset_pointer,
            "timestamp_column": "timestamp",
            "channels": channels,
        },
    })
    print('\n# explicit channels =', channels)
    print(json.dumps(r, indent=2, default=str))
else:
    print('No non-timestamp columns found for an explicit channel smoke test.')


### `characterize_series`

**Input:** the same file pointer and timestamp column.

**Expect:** a characterization result describing the series shape / grouping behavior.

In [ ]:
r = run_tsfm_tool("tsfm.characterize_series", {
    "dataset_path": smoke_pointer,
    "timestamp_column": "timestamp",
})
print(json.dumps(r, indent=2, default=str))


### `data_quality`

**Input:** the file pointer above, then a NaN-injected variant of the same file.

**Expect:** a quality report and a visible change when missing values are introduced.

In [ ]:
ensure_tu()
r = tu.run({
    'name': 'tsfm.data_quality',
    "arguments": {
        "dataset_path": dataset_pointer,
        "timestamp_column": "timestamp",
    },
})
print('# baseline')
print(json.dumps(r, indent=2, default=str))

r = tu.run({
    'name': 'tsfm.data_quality',
    "arguments": {
        "dataset_path": nan_pointer,
        "timestamp_column": "timestamp",
    },
})
print('\n# with missing values')
print(json.dumps(r, indent=2, default=str))


## 13. `model_template` - what shape is a card?

Start here. It reads nothing from the database; it just tells you the contract.

**Expect:** `required_fields = [model_id, description, task_ids]`, the 5 `pointer_choices`
(the ways a card can reference a model), and a filled `example` that registers as-is.

### A note on the response shape

FastMCP is **not uniform** about this, and you will see it below:

* a tool annotated `-> Union[X, ErrorResult]` comes back **wrapped**: `{"result": {...}}`
* `model_template`, annotated `-> ModelTemplateResult`, comes back **flat**

The raw response is printed as-is so the difference is visible.

In [ ]:
r = tu.run({
    'name': 'tsfm.model_template',
    'arguments': {},
})
print(json.dumps(r, indent=2, default=str))

## 12. `register_model` - add a real model

We register [`ibm-granite/granite-timeseries-ttm-r1`](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r1):
**805,280 parameters** - the first sub-1M "tiny" time-series foundation model.

A card is a **pointer**: `sktime_class` + `params.model_path` say how to build and load it.
No weights are stored here.

**Expect:** `status = registered`, `id = hub_ttm_r1`.

In [ ]:
R1  = 'ibm-granite/granite-timeseries-ttm-r1'
R2  = 'ibm-granite/granite-timeseries-ttm-r2'
TTM = 'sktime.forecasting.ttm.TinyTimeMixerForecaster'

r = tu.run({
    'name': 'tsfm.register_model',
    'arguments': {'model': {
        'model_id': 'hub_ttm_r1',
        'description': 'IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.',
        'task_ids': ['tsfm_forecasting'],
        'sktime_class': TTM,
        'params': {'model_path': R1},
        'hf_repo': R1,
        'model_family': 'TinyTimeMixer',
        'context_length': 512,
        'prediction_length': 96,
        'tags': ['foundation', 'tiny'],
        'version': '1',
    }},
})
print(json.dumps(r, indent=2, default=str))

### Try a bad card

**Expect:** an error naming both problems - `description` under 3 chars, `task_ids` empty.

In [ ]:
r = tu.run({
    'name': 'tsfm.register_model',
    'arguments': {'model': {'model_id': 'bad', 'description': 'ab', 'task_ids': []}},
})
print(json.dumps(r, indent=2, default=str))

## 18. `resolve_model` - can it actually load?

A read-only preflight. It checks the `sktime_class` is importable and reports where the
weights come from. It does **not** download anything.

**Expect:** `resolvable = true`, `weights_from` = the HF repo, `training_regime = zero_shot`.

In [ ]:
r = tu.run({
    'name': 'tsfm.resolve_model',
    'arguments': {'model_id': 'hub_ttm_r1'},
})
print(json.dumps(r, indent=2, default=str))

**Expect:** a clear `not found` error.

In [ ]:
r = tu.run({
    'name': 'tsfm.resolve_model',
    'arguments': {'model_id': 'no_such_model'},
})
print(json.dumps(r, indent=2, default=str))

## 15. `update_model` - patch a card

Merges `fields`, stamps `updated_at`, and re-validates. An invalid patch is rejected.

**Expect:** `domain = energy` and a fresh `updated_at`. Note the card comes back **flat**
here, whereas `register_model` nested it under `card`.

In [ ]:
r = tu.run({
    'name': 'tsfm.update_model',
    'arguments': {'model_id': 'hub_ttm_r1', 'fields': {'domain': 'finance'}},
})
print(json.dumps(r, indent=2, default=str))

## 14. `register_finetuned` - a fine-tune, with lineage

Say you fine-tuned TTM-R1 on Chiller 6 telemetry. This points a card at the checkpoint.

**Expect:** `sktime_class` **inherited** from the base, `params.model_path` = your checkpoint,
`provenance = finetuned`, `base_model_id = hub_ttm_r1`.

In [ ]:
r = tu.run({
    'name': 'tsfm.register_finetuned',
    'arguments': {
        'model_id': 'hub_ttm_ft',
        'checkpoint_path': '/artifacts/hub_ttm_ft',
        'base_model_id': 'hub_ttm_r1',
        'context_length': 512,
        'prediction_length': 96,
        'description': 'TTM-R1 fine-tuned on Chiller 6 telemetry',
        'domain': 'energy',
    },
})
print(json.dumps(r, indent=2, default=str))

### A known bug: an unknown base is accepted

Here the `base_model_id` does not exist. Watch what happens.

**Expect (this is the bug):** the card is **accepted**, and `sktime_class` silently falls back
to `TinyTimeMixerForecaster` - which may be the wrong architecture entirely. The tool
"succeeds" and the output looks plausible, which is what makes it dangerous. It is documented
in the tool's docstring as a caveat, and not yet fixed.

In [ ]:
r = tu.run({
    'name': 'tsfm.register_finetuned',
    'arguments': {
        'model_id': 'hub_orphan_ft',
        'checkpoint_path': '/artifacts/orphan',
        'base_model_id': 'does_not_exist',
        'context_length': 96,
        'prediction_length': 28,
        'description': 'fine-tune whose base does not exist',
    },
})
card = r.get('result', r)
print('base_model_id:', card.get('base_model_id'))
print('sktime_class :', card.get('sktime_class'), ' <-- silently defaulted')

## 17. `new_model_version` - supersede r1 with r2

HF's own R1 card declares `new_version: granite-timeseries-ttm-r2`, so let's follow it.

**Expect:** a new card `hub_ttm_r2` with `version = 2` and `supersedes = hub_ttm_r1`.
The predecessor flips to `status = superseded`.

In [ ]:
r = tu.run({
    'name': 'tsfm.new_model_version',
    'arguments': {
        'model_id': 'hub_ttm_r1',
        'fields': {'hf_repo': R2, 'params': {'model_path': R2}},
        'new_model_id': 'hub_ttm_r2',
    },
})
print(json.dumps(r, indent=2, default=str))

## 16. `deprecate_model` - retire r1

A **soft delete**: the document stays in CouchDB, it just drops out of active listings.
Reversible with `update_model(model_id, {'status': 'active'})`.

**Expect:** `status = deprecated` plus your `deprecation_reason`.

In [ ]:
r = tu.run({
    'name': 'tsfm.deprecate_model',
    'arguments': {'model_id': 'hub_ttm_r1', 'reason': 'superseded by R2 (per the HF card)'},
})
print(json.dumps(r, indent=2, default=str))

## What is left in the catalog?

**Expect:** `hub_ttm_r1` is **gone from the active list** (deprecated), while `hub_ttm_r2`,
the fine-tunes, and the seeded `ttm_96_28` remain.

In [ ]:
r = tu.run({'name': 'tsfm.list_models', 'arguments': {}})
models = r.get('result', r).get('models', [])
for m in sorted(models, key=lambda x: x['model_id']):
    print(f"  {m['model_id']:<18} {m.get('status'):<10} {m.get('provenance')}")

print('\nhub_ttm_r1 in the active list?', any(m['model_id'] == 'hub_ttm_r1' for m in models))

### The deprecated card is still in the database

It disappeared from `list_models` but the document is still there - that is what "soft delete"
means. You can see it in Fauxton too.

In [ ]:
try:
    doc = couch('/model_catalog/model:hub_ttm_r1')
    print('still in CouchDB ->', doc['_id'], '| status:', doc['status'])
    print('deprecation_reason:', doc.get('deprecation_reason'))
except Exception as e:
    print('(memory store, or not reachable):', e)

## Clean up

Close the stdio session. To reset the catalog, re-run `init_data.py --reset`.

Note the notebook is **not idempotent**: `register_model` overwrites by `model_id`, and
`new_model_version` bumps the version each run, so ids drift on repeat runs. Reset between runs.

In [ ]:
tu.close()
print('closed')